In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd

# Question 4

In [2]:
# parameters
g = 9.81

In [3]:
# Define Weibull parameters
shape = 2.2     # shape parameter (k)
loc = 15        # location parameter

wind = { "direction": ['W', 'NWN', 'NW'],
        "angle": [45, 22.5, 0],
        "fetch": [35430, 26450, 21990],
        "location": [16, 14, 15],
        "shape": [2.1, 2.2, 2.2],
        "depth": [5.21, 5.21, 5.21]}
wind = pd.DataFrame(wind)

In [4]:
speed95 = []
for i in range(3):
    speed95.append(round(stats.weibull_min.ppf(0.95, c=wind.loc[i, 'shape'], loc=wind.loc[i, 'location']), 2))

wind['speed95'] = speed95
display(wind)

,direction,angle,fetch,location,shape,depth,speed95
0,W,45.0,35430,16,2.1,5.21,17.69
1,NWN,22.5,26450,14,2.2,5.21,15.65
2,NW,0.0,21990,15,2.2,5.21,16.65


In [5]:
H_inf = 0.14
T_inf = 7.69
H_m0 = []
T_p =[]

for i in range(3):
    F_tilde = (g*wind.loc[i, 'fetch'])/(wind.loc[i, 'speed95']**2)
    d_tilde = (g*wind.loc[i, 'depth'])/(wind.loc[i, 'speed95']**2)
    H_tilde = H_inf*(np.tanh(0.343*(d_tilde**1.14))*np.tanh((4.41*(10**-4)*(F_tilde**0.79))/(np.tanh(0.343*(d_tilde**1.14))**2)))**0.572
    T_tilde = T_inf*(np.tanh(0.10*(d_tilde**2.01))*np.tanh((2.77*(10**-7)*(F_tilde**1.45))/(np.tanh(0.10*(d_tilde**2.01))**2)))**0.187
    H_m0.append(round(H_tilde*(wind.loc[i, 'speed95']**2)/g,2))
    T_p.append(round(T_tilde*(wind.loc[i, 'speed95']/g),2))

wind['H_m0'] = H_m0
wind['T_p'] = T_p

display(wind)

,direction,angle,fetch,location,shape,depth,speed95,H_m0,T_p
0,W,45.0,35430,16,2.1,5.21,17.69,0.74,4.56
1,NWN,22.5,26450,14,2.2,5.21,15.65,0.68,4.43
2,NW,0.0,21990,15,2.2,5.21,16.65,0.71,4.49


# Question 5

In [22]:
h = 5.21 # water depth in front of the dike
H = 7.0 # height of the dike

In [40]:
tan_a = 0.2 # slop of the dike
H_m0 = wind.loc[wind['direction'] == 'NW', 'H_m0'].values[0] # significant wave height of west
beta = wind.loc[wind['direction'] == 'NW', 'angle'].values[0]
T_p = wind.loc[wind['direction'] == 'NW', 'T_p'].values[0] # peak period of west
L_deep = g*T_p**2/(2*np.pi)
xi_m10 = tan_a/((H_m0/L_deep)**0.5)
R_C = H - h # freeboard of the dike
gamma_b = 1.0 # influence factor for a berm
gamma_f = 1.0 # influence factor for roughness elements on the slope
gamma_runup = 1-0.0022*beta
gamma_overtop =1-0.0033*beta
gamma_nu = 1.0 # influence factor for a wall at the end of a slope

In [41]:
gamma_beta = gamma_overtop
q = (0.026/np.sqrt(tan_a)) * gamma_b * xi_m10 * np.exp(-(2.5*(R_C/(xi_m10*H_m0*gamma_b*gamma_f*gamma_beta*gamma_nu)))**1.3) * np.sqrt(g* (H_m0**3))
print(f'The overtopping discharge is {q*1000:.3f} l/m/s per meter of dike length.')

The overtopping discharge is 0.077 l/m/s per meter of dike length.


In [42]:
q=5 / 1000
RC = ((xi_m10 * H_m0 * gamma_b * gamma_f * gamma_beta * gamma_nu) / 2.5) * ((-np.log(q*np.sqrt(tan_a)/(0.026*gamma_b*xi_m10*np.sqrt(g*(H_m0**3)))))**(1/1.3))
print(f'The freeboard needed is {RC:.2f} m')


The freeboard needed is 0.96 m


# Question 7

In [59]:
gamma_w = 10030 # unit weight of the water
gamma_s = 16000 # unit weight of the submerged particle
d70 = 2.8e-4 # 70%-fractile of grain size distribution
d70_m = 2.08e-4 # Reference value of 70%-fractile of grain size dis-tribution
L = 39.39 + 5 # piping length
H = 5.21 # water level at the foreside of the dike
tan_theta = 1/3.3 # slope of the dike
k = 7.52e-4 # hydraulic conductivity of the auqifer
D = 6.0 # thickness of the aquifer
eta = 0.25 # Drag factor coefficient 
m_p = 1 # Model factor piping
nu = 1.33e-6 # Kinematic viscosity 
g = 9.81 # Gravitational acceleration
h_b = 0 # water level inside the dike
d = 2.5 - 0.5 # impermeable sand layer at the sand boil exit point


In [60]:
def Limit_state_Sellmeijer(L, H, d70_m, d70, tan_theta, k, D, eta, m_p, nu, g, h_b, d):
    F_R = eta*(gamma_s/gamma_w)*tan_theta
    F_S = (d70_m / ((nu*k*L/g)**(1/3))) * ((d70/d70_m)**0.4)
    F_G = 0.91 * (D/L) ** ((0.28 / (((D/L)**0.28)-1))+0.04)
    H_c = m_p * F_R * F_S * F_R * L
    Z = H_c - (H - h_b - d*0.3)
    return Z, H_c

In [61]:
Z, H_c = Limit_state_Sellmeijer(L, H, d70_m, d70, tan_theta, k, D, eta, m_p, nu, g, h_b, d)

print(f'The critical hydraulic head difference is {H_c:.2f} m.')
print(f'The limit state {Z:.2f} m.')

The critical hydraulic head difference is 0.09 m.
The limit state -4.52 m.
